# Ablation training loop (local / VSCode)

Local version of kaggle notebook for the 4 feature-ablations
(`sp2020_CZ`, `bpi_2012_CZ`, `bpi_2013_CZ`, `BPI20_RequestForPayment_CZ`).



In [1]:
import torch
import pickle
import pandas as pd
import numpy as np 
import json
import os
import sys
import time
import random
from tqdm.auto import tqdm

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

root_path = os.getcwd()
sys.path.insert(0, os.path.join(root_path, "data"))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [3]:
from pipeline import HGNN, train_hgnn, test_hgnn, get_feature_variants, convert_feature_name

In [2]:
dataset = "small_log_CZ"

In [ ]:
data_dir = os.path.join(root_path, "data")
data_dir_processed = os.path.join(root_path, "data", "datasets", "comuzzi", "_processed")
data_dir_ablation = os.path.join(root_path, "data", "datasets", "ablation")
hyp_opt_dir = os.path.join(root_path, "results_CAISE", "hyp_opt")
results_root = os.path.join(root_path, "results_CAISE", "ablation", dataset) + "/"

os.makedirs(results_root, exist_ok=True)

with open(os.path.join(data_dir, "dataset_features.json")) as f:
    all_dataset_info = json.load(f)
dataset_info = all_dataset_info[dataset]
print(dataset_info)

tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv")

configs = pd.read_csv(f"{hyp_opt_dir}/{dataset}_CONFIGS.csv")
configs = configs.sort_values(by="AVG_total_loss")
best_row = configs.iloc[0]
best_parameters = {
    "hid": int(best_row["hid"]),
    "layers": int(best_row["layers"]),
    "lr": float(best_row["lr"]),
    "batch_size": int(best_row["batch_size"]),
    "weight_decay": float(best_row["weight_decay"]),
    "aggregation": best_row["aggregation"],
}
print(best_parameters)

In [ ]:
nan_methods = ["odd", "even", "random", "window", "attr_level"]
MISSING_VALUE = "MISSING_VALUE"

variants = get_feature_variants(dataset_info)
print(f"{len(variants)} variants to train for {dataset}")

## Preliminary Assessment

In [ ]:
expected_files = [f"{split}_V2_repair.pkl" for split in ["TRAIN", "VALID", "TEST"]] + \
                  [f"TEST_V2_repair_{t}.pkl" for t in nan_methods]

print(f"Checking presence of ablated graphs for: {dataset}...\n")

missing = []
for excluded_feature, cat, num in variants:
    safe_feature = convert_feature_name(excluded_feature)
    variant_dir = os.path.join(data_dir_ablation, dataset, f"ablate_{safe_feature}")
    ok = os.path.isdir(variant_dir) and all(
        os.path.exists(os.path.join(variant_dir, fname)) for fname in expected_files
    )
    print(f"  {'OK  ' if ok else 'MISS'}  ablate_{safe_feature}  ->  {variant_dir}")
    if not ok:
        missing.append(excluded_feature)

print(f"\n{len(variants) - len(missing)}/{len(variants)} found on {len(variants)} total")
if missing:
    print("Still missing (run Create_ablated_graphs.ipynb for these first):")
    for m in missing:
        print(f"  - {m}")

## Resume

In [ ]:
summary_path = f"{results_root}ablation_summary.csv"
summary_rows = pd.read_csv(summary_path).to_dict("records") if os.path.exists(summary_path) else []
already_done_features = {row["excluded_feature"] for row in summary_rows}
print(f"Features already completed: {sorted(already_done_features)}")

## Training

In [ ]:

from torch_geometric.transforms import ToUndirected      

transform = ToUndirected()                               

def create_df(results):
    res = {}
    for k in results[0]:
        res[k] = [x[k] for x in results]
    res = pd.DataFrame(data=res)
    return res, res.mean(), res.std()

elapsed_log = []

for excluded_feature, categorical_columns, real_value_columns in tqdm(variants, desc=f"Variants of {dataset}"):

    safe_feature = convert_feature_name(excluded_feature)

    if excluded_feature in already_done_features:
        tqdm.write(f"Variant ablate_{safe_feature} already completed, skipping.")
        continue

    t0 = time.time()
    try:
        variant_input_dir = os.path.join(data_dir_ablation, dataset, f"ablate_{safe_feature}")
        tqdm.write(f"\n=== Variant: ablate_{safe_feature} ===")

        with open(os.path.join(variant_input_dir, "TRAIN_V2_repair.pkl"), "rb") as f:
            X_train = pickle.load(f)
        with open(os.path.join(variant_input_dir, "VALID_V2_repair.pkl"), "rb") as f:
            X_valid = pickle.load(f)

        # NUOVO: grafi bidirezionali, come main_notebook_version.ipynb (cella 14)
        with torch.no_grad():
            for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
            for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])

        # edge_types calcolato DOPO il transform (come main_notebook_version.ipynb,
        # cella 15): altrimenti mancherebbero le convoluzioni per gli archi rev_*
        edge_types = set()
        for g in X_train + X_valid:
            _, e = g.metadata()
            edge_types.update(e)
        edge_types = list(edge_types)
        tqdm.write(f"  edge types: {len(edge_types)}")      # <-- NUOVO (verifica)

        list_unique = {k: list(tab_all[k].unique()) + [MISSING_VALUE] for k in categorical_columns}
        outputcat = {k: len(list_unique[k]) for k in list_unique}
        outputreal = real_value_columns

        res = {}
        for run in tqdm(range(10), desc=f"Run (ablate_{safe_feature})", leave=False):
            net = train_hgnn(
                best_parameters, outputcat, outputreal,
                edge_types=edge_types, X_train=X_train, X_valid=X_valid, device=device,
                epochs=100
            )
            for test_type in nan_methods:
                with open(os.path.join(variant_input_dir, f"TEST_V2_repair_{test_type}.pkl"), "rb") as f:
                    X = pickle.load(f)

                # NUOVO: anche i grafi di test, come fa test_multi in
                # main_notebook_version.ipynb (cella 35)
                with torch.no_grad():
                    for i in range(len(X)):
                        X[i] = transform(X[i])

                if test_type not in res:
                    res[test_type] = []
                res[test_type].append(test_hgnn(net, outputcat, outputreal, device=device, test_graphs=X))

        variant_dir = f"{results_root}ablation_{safe_feature}/"
        os.makedirs(variant_dir, exist_ok=True)

        for test_type in nan_methods:
            results_table, means, stds = create_df(res[test_type])
            results_table.to_csv(f"{variant_dir}{test_type}_V2_RESULTS_END.csv", sep=",", index=False)
            mean_std = pd.DataFrame(data={"mean": means, "std": stds})
            mean_std.to_csv(f"{variant_dir}{test_type}_V2_MEAN_STD_END.csv", sep=",")
            row = {"excluded_feature": excluded_feature, "test_type": test_type}
            row.update(means.to_dict())
            summary_rows.append(row)

        pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
        already_done_features.add(excluded_feature)

        dt = (time.time() - t0) / 3600
        elapsed_log.append((excluded_feature, dt))
        remaining = len([f for f, _, _ in variants if f not in already_done_features])
        tqdm.write(f"[ok] ablate_{safe_feature} in {dt:.2f}h — {remaining} variants left, ~{remaining*dt:.1f}h")

        del X_train, X_valid

    except Exception as e:
        tqdm.write(f"ERROR on variant ablate_{safe_feature}: {type(e).__name__}: {e}")
        tqdm.write("Continuing with the next variant...")

print("\nCycle finished.")
for f, dt in elapsed_log:
    print(f"  {f:22s} {dt:.2f}h")
if elapsed_log:
    print(f"  session total: {sum(d for _, d in elapsed_log):.2f}h")

## Recap

In [ ]:
summary = pd.read_csv(summary_path)
n_done = summary["excluded_feature"].nunique()
print(f"{n_done}/{len(variants)} variants completed\n")
summary